
# 03 — Gold Layer Aggregation

## Objective
Build business-ready Delta datasets from the validated Silver layer.

## Gold datasets
1. trip_summary
2. route_daily_performance
3. vehicle_daily_scorecard
4. cold_chain_daily_summary

## Design principles
- Preserve source data and lineage.
- Avoid silently resolving source disagreements.
- Handle missing values explicitly.
- Use incremental and idempotent processing where practical.
- Keep anomaly labels separate from anomaly detection inputs.



## Step 1 — Load Silver Data

Load the validated Silver Delta datasets required for the trip summary.

We will use:
- Trips: actual trip information
- Trip plans: planned trip information
- Vehicles: vehicle attributes
- Drivers: driver attributes
- Shipments: cargo and temperature requirements
- Routes: route metadata

The source datasets will remain unchanged.


In [0]:

# Define storage paths and load the Silver datasets needed for trip summary.

from pyspark.sql import functions as F

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

silver_paths = {
    "trips": storage_path + "silver/trips/",
    "trip_plans": storage_path + "silver/master_data/trip_plans/",
    "vehicles": storage_path + "silver/master_data/vehicles/",
    "drivers": storage_path + "silver/master_data/drivers/",
    "shipments": storage_path + "silver/master_data/shipments/",
    "routes": storage_path + "silver/master_data/routes/"
}

trips_df = spark.read.format("delta").load(silver_paths["trips"])
trip_plans_df = spark.read.format("delta").load(silver_paths["trip_plans"])
vehicles_df = spark.read.format("delta").load(silver_paths["vehicles"])
drivers_df = spark.read.format("delta").load(silver_paths["drivers"])
shipments_df = spark.read.format("delta").load(silver_paths["shipments"])
routes_df = spark.read.format("delta").load(silver_paths["routes"])

# Display the schema of the main trip dataset.
trips_df.printSchema()


root
 |-- trip_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- planned_start: timestamp (nullable = true)
 |-- planned_end: timestamp (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- actual_start: timestamp (nullable = true)
 |-- actual_end: timestamp (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fuel_used_liters: double (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)




## Step 2 — Build Trip Summary

Join operational trips with trip plans and master data.

- Preserve every trip from the operational trips dataset.
- Keep planned and operational statuses separate.
- Preserve source disagreements for later investigation.
- Retain null values when the source data is incomplete.


In [0]:

# Build the trip summary by joining trips with plans and master data.

trip_summary_df = (
    trips_df.alias("t")
    .join(
        trip_plans_df.select(
            "trip_id",
            F.col("trip_status").alias("plan_status"),
            F.col("planned_start").alias("plan_planned_start"),
            F.col("planned_end").alias("plan_planned_end")
        ).alias("p"),
        on="trip_id",
        how="left"
    )
    .join(
        vehicles_df.select(
            "vehicle_id",
            "vehicle_type",
            "model",
            "fuel_type",
            "capacity_kg"
        ),
        on="vehicle_id",
        how="left"
    )
    .join(
        drivers_df.select(
            "driver_id",
            "license_category",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
    .join(
        shipments_df.select(
            "shipment_id",
            "cargo_type",
            "required_temp_min_c",
            "required_temp_max_c",
            F.col("delivery_status").alias("shipment_status")
        ),
        on="shipment_id",
        how="left"
    )
    .join(
        routes_df.select(
            "route_id",
            "route_name",
            "origin_name",
            "destination_name",
            "planned_distance_km",
            "planned_duration_min"
        ),
        on="route_id",
        how="left"
    )
)

# Check the resulting row count and uniqueness of trip IDs.
trip_summary_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("trip_id").alias("unique_trip_ids"),
    F.sum(F.when(F.col("plan_status").isNull(), 1).otherwise(0))
        .alias("missing_plan_matches"),
    F.sum(F.when(F.col("vehicle_type").isNull(), 1).otherwise(0))
        .alias("missing_vehicle_matches"),
    F.sum(F.when(F.col("license_category").isNull(), 1).otherwise(0))
        .alias("missing_driver_matches"),
    F.sum(F.when(F.col("cargo_type").isNull(), 1).otherwise(0))
        .alias("missing_shipment_matches"),
    F.sum(F.when(F.col("route_name").isNull(), 1).otherwise(0))
        .alias("missing_route_matches")
).show(truncate=False)

# Inspect representative records before saving.
display(trip_summary_df.limit(10))


+----------+---------------+--------------------+-----------------------+----------------------+------------------------+---------------------+
|total_rows|unique_trip_ids|missing_plan_matches|missing_vehicle_matches|missing_driver_matches|missing_shipment_matches|missing_route_matches|
+----------+---------------+--------------------+-----------------------+----------------------+------------------------+---------------------+
|5000      |5000           |0                   |0                      |0                     |0                       |0                    |
+----------+---------------+--------------------+-----------------------+----------------------+------------------------+---------------------+



route_id,shipment_id,driver_id,vehicle_id,trip_id,planned_start,planned_end,trip_status,actual_start,actual_end,distance_km,fuel_used_liters,origin,destination,source_file,ingested_at,plan_status,plan_planned_start,plan_planned_end,vehicle_type,model,fuel_type,capacity_kg,license_category,employment_status,cargo_type,required_temp_min_c,required_temp_max_c,shipment_status,route_name,origin_name,destination_name,planned_distance_km,planned_duration_min
RTE-001,SHP-00001,DRV-00001,VEH-00001,TRIP-00001-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,cairo_ring,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,cairo_ring,Cairo,Cairo Ring,null,null
RTE-010,SHP-00002,DRV-00001,VEH-00002,TRIP-00002-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,north_cairo,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,north_cairo,Cairo,North Cairo,null,null
RTE-001,SHP-00003,DRV-00001,VEH-00003,TRIP-00003-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,cairo_ring,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,cairo_ring,Cairo,Cairo Ring,null,null
RTE-007,SHP-00004,DRV-00002,VEH-00004,TRIP-00004-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,beni_suef,"List(29.0661, 31.0994)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,beni_suef,Cairo,Beni Suef,null,null
RTE-007,SHP-00005,DRV-00002,VEH-00005,TRIP-00005-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,beni_suef,"List(29.0661, 31.0994)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,van,Transit Van,diesel,1500.0,professional_heavy,active,general_goods,null,null,planned,beni_suef,Cairo,Beni Suef,null,null
RTE-009,SHP-00006,DRV-00002,VEH-00006,TRIP-00006-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,port_said,"List(31.2565, 32.2841)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,port_said,Cairo,Port Said,null,null
RTE-006,SHP-00007,DRV-00002,VEH-00007,TRIP-00007-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,fayoum,"List(29.3084, 30.8428)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,truck,Heavy Duty Truck,diesel,12000.0,professional_heavy,active,general_goods,null,null,planned,fayoum,Cairo,Fayoum,null,null
RTE-003,SHP-00008,DRV-00003,VEH-00008,TRIP-00008-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,alex_desert,"List(31.2001, 29.9187)",trips.json,2026-10-26T23:40:50.754Z,planned,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,refrigerated,Refrigerated Truck,diesel,8000.0,professional_light,active,temperature_controlled_food,-5.0,5.0,planned,alex_desert,Cairo,Alex Desert,null,null
RTE-009,SHP-00009,DRV-00003,VEH-00009,TRIP-00009-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,port_said,"List(31.2565, 32.2841)",trip


## Step 3 — Finalize and Save Trip Summary

Select business-relevant columns, preserve both operational and planned values, and save the result as a Delta dataset.

The output will be stored at `gold/trip_summary/`.

In [0]:

# Select the final business columns and save the trip summary as Delta.

trip_summary_final_df = trip_summary_df.select(
    "trip_id",
    "vehicle_id",
    "driver_id",
    "shipment_id",
    "route_id",

    F.col("trip_status").alias("operational_status"),
    F.col("plan_status"),

    F.col("planned_start").alias("trip_planned_start"),
    F.col("planned_end").alias("trip_planned_end"),
    F.col("plan_planned_start"),
    F.col("plan_planned_end"),

    "actual_start",
    "actual_end",
    "distance_km",
    "fuel_used_liters",

    "vehicle_type",
    "model",
    "fuel_type",
    "capacity_kg",

    "license_category",
    "employment_status",

    "cargo_type",
    "required_temp_min_c",
    "required_temp_max_c",
    "shipment_status",

    "route_name",
    "origin_name",
    "destination_name",
    "planned_distance_km",
    "planned_duration_min",

    "origin",
    "destination",
    "source_file",
    "ingested_at"
).withColumn(
    "planned_end_difference_minutes",
    (
        F.col("trip_planned_end").cast("long")
        - F.col("plan_planned_end").cast("long")
    ) / 60.0
)

# Save the Gold dataset to its dedicated Delta path.
gold_trip_summary_path = storage_path + "gold/trip_summary/"

trip_summary_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_trip_summary_path)

print("gold.trip_summary saved successfully.")
print(f"Output path: {gold_trip_summary_path}")


gold.trip_summary saved successfully.
Output path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/gold/trip_summary/


In [0]:

# Validate the saved Gold Delta dataset without rereading the source datasets.

saved_trip_summary_df = (
    spark.read.format("delta").load(gold_trip_summary_path)
)

saved_trip_summary_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("trip_id").alias("unique_trip_ids"),
    F.sum(
        F.when(F.col("operational_status") != F.col("plan_status"), 1)
        .otherwise(0)
    ).alias("status_mismatches"),
    F.sum(
        F.when(F.col("planned_end_difference_minutes").isNotNull(), 1)
        .otherwise(0)
    ).alias("rows_with_time_difference")
).show(truncate=False)


+----------+---------------+-----------------+-------------------------+
|total_rows|unique_trip_ids|status_mismatches|rows_with_time_difference|
+----------+---------------+-----------------+-------------------------+
|5000      |5000           |5000             |5000                     |
+----------+---------------+-----------------+-------------------------+




## Step 4 — Route Daily Performance

Aggregate trip-level data by route and planned start date.

Metrics:
- Total trips
- Trips in progress
- Planned trips
- Completed trips
- Trips with actual distance available
- Total actual distance where available

Missing distance values will remain missing and will not be replaced with zero.


In [0]:

# Build daily route performance metrics from the saved Gold trip summary.

trip_summary_gold_df = (
    spark.read.format("delta").load(gold_trip_summary_path)
)

route_daily_performance_df = (
    trip_summary_gold_df
    .withColumn("performance_date", F.to_date("trip_planned_start"))
    .groupBy("route_id", "route_name", "performance_date")
    .agg(
        F.countDistinct("trip_id").alias("total_trips"),

        F.sum(
            F.when(F.col("operational_status") == "in_progress", 1)
            .otherwise(0)
        ).alias("in_progress_trips"),

        F.sum(
            F.when(F.col("plan_status") == "planned", 1)
            .otherwise(0)
        ).alias("planned_trips"),

        F.sum(
            F.when(F.col("operational_status") == "completed", 1)
            .otherwise(0)
        ).alias("completed_trips"),

        F.sum(
            F.when(F.col("distance_km").isNotNull(), 1)
            .otherwise(0)
        ).alias("trips_with_actual_distance"),

        F.sum("distance_km").alias("total_actual_distance_km")
    )
)

# Inspect the route-level daily results before saving.
display(
    route_daily_performance_df
    .orderBy("performance_date", "route_id")
)


route_id,route_name,performance_date,total_trips,in_progress_trips,planned_trips,completed_trips,trips_with_actual_distance,total_actual_distance_km
RTE-001,cairo_ring,2026-10-01,487,487,487,0,0,null
RTE-002,new_cairo,2026-10-01,488,488,488,0,0,null
RTE-003,alex_desert,2026-10-01,500,500,500,0,0,null
RTE-004,ismailia,2026-10-01,518,518,518,0,0,null
RTE-005,suez,2026-10-01,475,475,475,0,0,null
RTE-006,fayoum,2026-10-01,521,521,521,0,0,null
RTE-007,beni_suef,2026-10-01,497,497,497,0,0,null
RTE-008,assuit,2026-10-01,477,477,477,0,0,null
RTE-009,port_said,2026-10-01,510,510,510,0,0,null
RTE-010,north_cairo,2026-10-01,527,527,527,0,0,null


In [0]:

# Save the validated daily route performance aggregation as Delta.

gold_route_daily_path = storage_path + "gold/route_daily_performance/"

route_daily_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_route_daily_path)

print("gold.route_daily_performance saved successfully.")
print(f"Output path: {gold_route_daily_path}")


gold.route_daily_performance saved successfully.
Output path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/gold/route_daily_performance/



## Step 5 — Vehicle Daily Scorecard

Aggregate Silver telemetry by vehicle and event date.

Metrics:
- Telemetry event count
- Average and maximum speed
- Average engine temperature
- Average RPM
- Average recorded fuel rate

The fuel-rate metric represents recorded sensor readings, not total fuel consumed in liters.


In [0]:
# Define storage paths and add the streaming silver path
storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

silver_paths = {
    "telemetry": storage_path + "silver/telemetry/",
    "telemetry_stream": storage_path + "silver/telemetry_stream/", # Added streaming path
    "trips": storage_path + "silver/trips/",
    "trip_plans": storage_path + "silver/master_data/trip_plans/",
    "vehicles": storage_path + "silver/master_data/vehicles/",
    "drivers": storage_path + "silver/master_data/drivers/",
    "shipments": storage_path + "silver/master_data/shipments/",
    "routes": storage_path + "silver/master_data/routes/",
    "anomaly_labels": storage_path + "silver/anomaly_labels/"
}

print("Streaming telemetry source:", silver_paths["telemetry_stream"])

abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/telemetry/


In [0]:
# Read only the incremental micro-batch from telemetry_stream
telemetry_gold_source_df = (
    spark.read.format("delta").load(silver_paths["telemetry_stream"])
)

vehicle_daily_scorecard_df = (
    telemetry_gold_source_df
    .withColumn("scorecard_date", F.to_date("event_timestamp"))
    .groupBy("vehicle_id", "scorecard_date")
    .agg(
        F.count("*").alias("telemetry_event_count"),
        F.avg("speed_kph").alias("avg_speed_kph"),
        F.max("speed_kph").alias("max_speed_kph"),
        F.avg("engine_temp_c").alias("avg_engine_temp_c"),
        F.avg("rpm").alias("avg_rpm"),
        F.avg("fuel_rate_lph").alias("avg_fuel_rate_lph")
    )
)

print(f"Incremental vehicle records to merge: {vehicle_daily_scorecard_df.count()}")
display(vehicle_daily_scorecard_df.limit(10))

Daily vehicle records: 10000


vehicle_id,scorecard_date,telemetry_event_count,avg_speed_kph,max_speed_kph,avg_engine_temp_c,avg_rpm,avg_fuel_rate_lph
VEH-00001,2026-10-01,1440,46.6323125,60.36,85.41124305555557,2039.5479166666667,4.411095138888888
VEH-00002,2026-10-01,1440,44.14672916666667,60.27,94.59806944444445,1968.1145833333333,7.113602083333335
VEH-00003,2026-10-01,1440,60.45441666666667,76.43,96.47602777777779,2437.6,9.232005555555554
VEH-00004,2026-10-01,1440,37.32196527777777,47.93,88.83759027777778,1781.4798611111112,6.43815625
VEH-00005,2026-10-01,1440,46.9959861111111,61.62,92.05581250000002,1858.3770833333333,5.531570138888888
VEH-00006,2026-10-01,1440,50.35358333333333,64.17,88.52642361111111,2137.557638888889,6.765315972222221
VEH-00007,2026-10-01,1440,61.91158333333332,77.52,91.84920138888891,2463.1868055555556,9.280524999999999
VEH-00008,2026-10-01,1440,36.393833333333326,46.62,82.4305902777778,1602.8590277777778,4.498972916666666
VEH-00009,2026-10-01,1440,45.56142361111111,60.88,86.15728472222226,1828.7833333333333,5.194969444444443
VEH-00010,2026-10-01,1440,42.94902777777778,55.66,95.43138194444442,1758.1763888888888,4.305334722222224


In [0]:
from delta.tables import DeltaTable

gold_vehicle_scorecard_path = storage_path + "gold/vehicle_daily_scorecard/"

# Upsert (MERGE) daily vehicle scorecards
if DeltaTable.isDeltaTable(spark, gold_vehicle_scorecard_path):
    target_scorecard = DeltaTable.forPath(spark, gold_vehicle_scorecard_path)
    (
        target_scorecard.alias("target")
        .merge(
            vehicle_daily_scorecard_df.alias("source"),
            "target.vehicle_id = source.vehicle_id AND target.scorecard_date = source.scorecard_date"
        )
        .whenMatchedUpdate(set={
            "telemetry_event_count": "target.telemetry_event_count + source.telemetry_event_count",
            "avg_speed_kph": "(target.avg_speed_kph + source.avg_speed_kph) / 2",
            "max_speed_kph": "greatest(target.max_speed_kph, source.max_speed_kph)",
            "avg_engine_temp_c": "(target.avg_engine_temp_c + source.avg_engine_temp_c) / 2",
            "avg_rpm": "(target.avg_rpm + source.avg_rpm) / 2",
            "avg_fuel_rate_lph": "(target.avg_fuel_rate_lph + source.avg_fuel_rate_lph) / 2"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("gold.vehicle_daily_scorecard merged successfully.")
else:
    vehicle_daily_scorecard_df.write.format("delta").mode("overwrite").save(gold_vehicle_scorecard_path)
    print("gold.vehicle_daily_scorecard initialized successfully.")

gold.vehicle_daily_scorecard saved successfully.
Output path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/gold/vehicle_daily_scorecard/



## Step 6 — Cold Chain Daily Summary

Analyze cargo temperature readings by vehicle, shipment, and event date.

Metrics:
- Temperature reading count
- Average, minimum, and maximum cargo temperature
- Readings below or above the shipment's required temperature range
- Missing temperature-limit indicator

Only shipments with known temperature limits will be evaluated for out-of-range readings. Missing limits will not be inferred.


In [0]:
# Read from telemetry_stream instead of historical telemetry
telemetry_cold_df = (
    spark.read.format("delta").load(silver_paths["telemetry_stream"])
    .filter(F.col("cargo_temp_c").isNotNull())
    .select(
        "vehicle_id",
        "shipment_id",
        "event_timestamp",
        "cargo_temp_c"
    )
)

cold_chain_plans_df = (
    spark.read.format("delta").load(silver_paths["trip_plans"])
    .select("vehicle_id", "shipment_id")
    .dropDuplicates(["vehicle_id", "shipment_id"])
)

cold_chain_shipments_df = (
    spark.read.format("delta").load(silver_paths["shipments"])
    .select(
        "shipment_id",
        "required_temp_min_c",
        "required_temp_max_c"
    )
)

cold_chain_joined_df = (
    telemetry_cold_df
    .join(cold_chain_plans_df, on=["vehicle_id", "shipment_id"], how="inner")
    .join(cold_chain_shipments_df, on="shipment_id", how="left")
    .withColumn("summary_date", F.to_date("event_timestamp"))
)

cold_chain_daily_summary_df = (
    cold_chain_joined_df
    .groupBy("vehicle_id", "shipment_id", "summary_date")
    .agg(
        F.count("*").alias("temperature_reading_count"),
        F.avg("cargo_temp_c").alias("avg_cargo_temp_c"),
        F.min("cargo_temp_c").alias("min_cargo_temp_c"),
        F.max("cargo_temp_c").alias("max_cargo_temp_c"),
        F.max(
            F.when(
                F.col("required_temp_min_c").isNull() | F.col("required_temp_max_c").isNull(), 1
            ).otherwise(0)
        ).alias("temperature_limits_missing"),
        F.sum(
            F.when(
                F.col("required_temp_min_c").isNotNull() & (F.col("cargo_temp_c") < F.col("required_temp_min_c")), 1
            ).otherwise(0)
        ).alias("readings_below_required_min"),
        F.sum(
            F.when(
                F.col("required_temp_max_c").isNotNull() & (F.col("cargo_temp_c") > F.col("required_temp_max_c")), 1
            ).otherwise(0)
        ).alias("readings_above_required_max"),
        F.sum(
            F.when(
                F.col("required_temp_min_c").isNotNull() & F.col("required_temp_max_c").isNotNull(), 1
            ).otherwise(0)
        ).alias("readings_with_known_limits")
    )
)

vehicle_id,shipment_id,summary_date,temperature_reading_count,avg_cargo_temp_c,min_cargo_temp_c,max_cargo_temp_c,temperature_limits_missing,readings_below_required_min,readings_above_required_max,readings_with_known_limits
VEH-00340,SHP-00340,2026-10-01,1440,1.2684791666666668,-0.82,9.9,0,0,29,1440
VEH-03106,SHP-03106,2026-10-01,1440,-0.12016666666666669,-0.76,9.92,0,0,29,1440
VEH-03434,SHP-03434,2026-10-01,1440,-0.46043750000000006,-1.1,9.97,0,0,27,1440
VEH-00045,SHP-00045,2026-10-01,1440,-0.01717361111111108,-0.68,9.98,0,0,31,1440
VEH-00370,SHP-00370,2026-10-01,1440,1.0834305555555557,0.05,9.94,0,0,24,1440
VEH-04059,SHP-04059,2026-10-01,1440,0.9947013888888893,-0.74,9.91,0,0,28,1440
VEH-04557,SHP-04557,2026-10-01,1440,-0.8188333333333331,-1.43,9.91,0,0,18,1440
VEH-01829,SHP-01829,2026-10-01,1440,1.289826388888889,0.38,9.99,0,0,35,1440
VEH-02010,SHP-02010,2026-10-01,1440,-0.7903055555555557,-1.39,9.92,0,0,22,1440
VEH-00160,SHP-00160,2026-10-01,1440,1.9081180555555557,0.13,9.72,0,0,17,1440


In [0]:
from delta.tables import DeltaTable

gold_cold_chain_path = storage_path + "gold/cold_chain_daily_summary/"

# Upsert (MERGE) cold-chain metrics
if DeltaTable.isDeltaTable(spark, gold_cold_chain_path):
    target_cold = DeltaTable.forPath(spark, gold_cold_chain_path)
    (
        target_cold.alias("target")
        .merge(
            cold_chain_daily_summary_df.alias("source"),
            "target.vehicle_id = source.vehicle_id AND target.shipment_id = source.shipment_id AND target.summary_date = source.summary_date"
        )
        .whenMatchedUpdate(set={
            "temperature_reading_count": "target.temperature_reading_count + source.temperature_reading_count",
            "avg_cargo_temp_c": "(target.avg_cargo_temp_c + source.avg_cargo_temp_c) / 2",
            "min_cargo_temp_c": "least(target.min_cargo_temp_c, source.min_cargo_temp_c)",
            "max_cargo_temp_c": "greatest(target.max_cargo_temp_c, source.max_cargo_temp_c)",
            "temperature_limits_missing": "greatest(target.temperature_limits_missing, source.temperature_limits_missing)",
            "readings_below_required_min": "target.readings_below_required_min + source.readings_below_required_min",
            "readings_above_required_max": "target.readings_above_required_max + source.readings_above_required_max",
            "readings_with_known_limits": "target.readings_with_known_limits + source.readings_with_known_limits"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("gold.cold_chain_daily_summary merged successfully.")
else:
    cold_chain_daily_summary_df.write.format("delta").mode("overwrite").save(gold_cold_chain_path)
    print("gold.cold_chain_daily_summary initialized successfully.")

gold.cold_chain_daily_summary saved successfully.
Output path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/gold/cold_chain_daily_summary/


In [0]:

# Inspect the saved Gold Delta datasets without recounting all rows.

gold_paths = {
    "trip_summary": storage_path + "gold/trip_summary/",
    "route_daily_performance": storage_path + "gold/route_daily_performance/",
    "vehicle_daily_scorecard": storage_path + "gold/vehicle_daily_scorecard/",
    "cold_chain_daily_summary": storage_path + "gold/cold_chain_daily_summary/"
}

for dataset_name, dataset_path in gold_paths.items():
    delta_detail = (
        spark.sql(
            f"DESCRIBE DETAIL delta.`{dataset_path}`"
        )
        .select("format", "numFiles", "sizeInBytes")
        .first()
    )

    print(
        f"{dataset_name}: "
        f"format={delta_detail['format']}, "
        f"files={delta_detail['numFiles']}, "
        f"size_MB={round(delta_detail['sizeInBytes'] / (1024 * 1024), 2)}"
    )


trip_summary: format=delta, files=1, size_MB=0.11
route_daily_performance: format=delta, files=1, size_MB=0.0
vehicle_daily_scorecard: format=delta, files=1, size_MB=0.35
cold_chain_daily_summary: format=delta, files=1, size_MB=0.06


In [0]:

# Calculate temperature compliance from the saved Gold dataset.

cold_chain_saved_df = (
    spark.read.format("delta")
    .load(gold_cold_chain_path)
)

cold_chain_kpi_df = (
    cold_chain_saved_df
    .agg(
        F.sum("readings_with_known_limits").alias("evaluable_readings"),
        F.sum("readings_below_required_min").alias("readings_below_min"),
        F.sum("readings_above_required_max").alias("readings_above_max")
    )
    .withColumn(
        "compliant_readings",
        F.col("evaluable_readings")
        - F.col("readings_below_min")
        - F.col("readings_above_max")
    )
    .withColumn(
        "temperature_compliance_pct",
        F.when(
            F.col("evaluable_readings") > 0,
            F.round(
                F.col("compliant_readings")
                / F.col("evaluable_readings") * 100,
                2
            )
        )
    )
)

display(cold_chain_kpi_df)


evaluable_readings,readings_below_min,readings_above_max,compliant_readings,temperature_compliance_pct
2940000,0,52660,2887340,98.21



## Step 7 — Trip Status Reconciliation

Compare operational trip records with planned trip records to investigate status and schedule mismatches before calculating the Trip Completion Rate.


In [0]:

# Compare trip statuses and planned end times across the two Silver datasets.

trips_status_check_df = (
    spark.read.format("delta")
    .load(silver_paths["trips"])
    .select(
        "trip_id",
        F.col("trip_status").alias("operational_status"),
        F.col("planned_start").alias("operational_planned_start"),
        F.col("planned_end").alias("operational_planned_end"),
        "actual_start",
        "actual_end"
    )
)

plans_status_check_df = (
    spark.read.format("delta")
    .load(silver_paths["trip_plans"])
    .select(
        "trip_id",
        F.col("trip_status").alias("plan_status"),
        F.col("planned_start").alias("plan_planned_start"),
        F.col("planned_end").alias("plan_planned_end")
    )
)

# Summarize the distinct status combinations without scanning telemetry.
status_reconciliation_df = (
    trips_status_check_df
    .join(plans_status_check_df, on="trip_id", how="inner")
    .groupBy("operational_status", "plan_status")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum(
            F.when(F.col("actual_start").isNotNull(), 1).otherwise(0)
        ).alias("trips_with_actual_start"),
        F.sum(
            F.when(F.col("actual_end").isNotNull(), 1).otherwise(0)
        ).alias("trips_with_actual_end"),
        F.countDistinct("trip_id").alias("unique_trips")
    )
)

display(status_reconciliation_df)


operational_status,plan_status,trip_count,trips_with_actual_start,trips_with_actual_end,unique_trips
in_progress,planned,5000,5000,0,5000


In [0]:

# Check whether operational timestamps support a completion-rate calculation.

trip_completion_evidence_df = (
    trips_status_check_df
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(
            F.when(F.col("actual_start").isNotNull(), 1).otherwise(0)
        ).alias("with_actual_start"),
        F.sum(
            F.when(F.col("actual_end").isNotNull(), 1).otherwise(0)
        ).alias("with_actual_end"),
        F.sum(
            F.when(
                F.col("actual_start").isNotNull()
                & F.col("actual_end").isNotNull(),
                1
            ).otherwise(0)
        ).alias("with_both_actual_timestamps"),
        F.sum(
            F.when(
                F.col("operational_status") == "completed",
                1
            ).otherwise(0)
        ).alias("marked_completed")
    )
)

display(trip_completion_evidence_df)


total_trips,with_actual_start,with_actual_end,with_both_actual_timestamps,marked_completed
5000,5000,0,0,0


In [0]:

# Add transparent reconciliation fields while preserving both source statuses.

trip_summary_reconciled_df = (
    spark.read.format("delta")
    .load(gold_trip_summary_path)
    .withColumn(
        "operational_evidence_status",
        F.when(
            F.col("actual_end").isNotNull(),
            F.lit("has_actual_end")
        ).when(
            F.col("actual_start").isNotNull(),
            F.lit("started_no_end_timestamp")
        ).otherwise(
            F.lit("no_actual_start_timestamp")
        )
    )
    .withColumn(
        "status_reconciliation",
        F.when(
            (F.col("operational_status") == "in_progress")
            & (F.col("plan_status") == "planned")
            & F.col("actual_start").isNotNull()
            & F.col("actual_end").isNull(),
            F.lit("planned_but_started_no_end")
        ).when(
            F.col("actual_end").isNotNull(),
            F.lit("actual_end_timestamp_available")
        ).otherwise(
            F.lit("needs_review")
        )
    )
)

# Review the classifications before replacing the Gold output.
display(
    trip_summary_reconciled_df
    .groupBy("operational_status", "plan_status",
             "operational_evidence_status", "status_reconciliation")
    .count()
)


operational_status,plan_status,operational_evidence_status,status_reconciliation,count
in_progress,planned,started_no_end_timestamp,planned_but_started_no_end,5000


In [0]:

# Explicitly add the two reconciliation columns to the existing Delta dataset.

spark.sql(f"""
ALTER TABLE delta.`{gold_trip_summary_path}`
ADD COLUMNS (
    operational_evidence_status STRING,
    status_reconciliation STRING
)
""")

print("Trip summary schema updated successfully.")


Trip summary schema updated successfully.


In [0]:

# Reload the updated Delta schema and merge the reconciliation values by trip_id.

from delta.tables import DeltaTable

trip_summary_target = DeltaTable.forPath(
    spark,
    gold_trip_summary_path
)

trip_summary_target.alias("target").merge(
    trip_summary_reconciled_df.select(
        "trip_id",
        "operational_evidence_status",
        "status_reconciliation"
    ).alias("source"),
    "target.trip_id = source.trip_id"
).whenMatchedUpdate(
    set={
        "operational_evidence_status": "source.operational_evidence_status",
        "status_reconciliation": "source.status_reconciliation"
    }
).execute()

print("Trip status reconciliation saved successfully.")


Trip status reconciliation saved successfully.


In [0]:

# Verify the saved reconciliation categories.

trip_summary_saved_df = (
    spark.read.format("delta")
    .load(gold_trip_summary_path)
)

display(
    trip_summary_saved_df
    .groupBy("operational_evidence_status", "status_reconciliation")
    .count()
)


operational_evidence_status,status_reconciliation,count
started_no_end_timestamp,planned_but_started_no_end,5000


In [0]:

# Calculate recorded trip status KPIs from the saved Gold dataset.

trip_kpis_df = (
    spark.read.format("delta")
    .load(gold_trip_summary_path)
    .agg(
        F.countDistinct("trip_id").alias("total_trips"),
        F.countDistinct(
            F.when(
                F.col("operational_status") == "completed",
                F.col("trip_id")
            )
        ).alias("recorded_completed_trips"),
        F.countDistinct(
            F.when(
                F.col("actual_start").isNotNull(),
                F.col("trip_id")
            )
        ).alias("trips_with_actual_start"),
        F.countDistinct(
            F.when(
                F.col("actual_end").isNotNull(),
                F.col("trip_id")
            )
        ).alias("trips_with_actual_end")
    )
    .withColumn(
        "recorded_completion_rate_pct",
        F.round(
            F.col("recorded_completed_trips")
            / F.col("total_trips") * 100,
            2
        )
    )
)

display(trip_kpis_df)


total_trips,recorded_completed_trips,trips_with_actual_start,trips_with_actual_end,recorded_completion_rate_pct
5000,0,5000,0,0.0


##Cold Chain Risk by Shipment

In [0]:

# Analyze cold-chain temperature risks using the saved Gold dataset only.

cold_chain_df = spark.read.format("delta").load(gold_cold_chain_path)

shipment_risk_df = (
    cold_chain_df
    .groupBy("shipment_id")
    .agg(
        F.sum("temperature_reading_count").alias("total_readings"),
        F.sum("readings_with_known_limits").alias("evaluable_readings"),
        F.sum("readings_above_required_max").alias("readings_above_max"),
        F.sum("readings_below_required_min").alias("readings_below_min"),
        F.max("temperature_limits_missing").alias("temperature_limits_missing")
    )
    .withColumn(
        "out_of_range_readings",
        F.col("readings_above_max") + F.col("readings_below_min")
    )
    .withColumn(
        "out_of_range_pct",
        F.when(
            F.col("evaluable_readings") > 0,
            F.round(
                F.col("out_of_range_readings") / F.col("evaluable_readings") * 100,
                2
            )
        )
    )
    .withColumn(
        "risk_status",
        F.when(
            F.col("evaluable_readings") == 0,
            "Cannot Evaluate"
        ).when(
            F.col("out_of_range_readings") > 0,
            "Temperature Risk"
        ).otherwise("Within Limits")
    )
)

display(
    shipment_risk_df
    .orderBy(
        F.col("out_of_range_readings").desc(),
        F.col("out_of_range_pct").desc_nulls_last()
    )
    .limit(10)
)


shipment_id,total_readings,evaluable_readings,readings_above_max,readings_below_min,temperature_limits_missing,out_of_range_readings,out_of_range_pct,risk_status
SHP-02320,2000,2000,59,0,0,59,2.95,Temperature Risk
SHP-04754,2000,2000,57,0,0,57,2.85,Temperature Risk
SHP-00281,2000,2000,56,0,0,56,2.8,Temperature Risk
SHP-03618,2000,2000,55,0,0,55,2.75,Temperature Risk
SHP-03666,2000,2000,54,0,0,54,2.7,Temperature Risk
SHP-02665,2000,2000,53,0,0,53,2.65,Temperature Risk
SHP-01757,2000,2000,52,0,0,52,2.6,Temperature Risk
SHP-04794,2000,2000,52,0,0,52,2.6,Temperature Risk
SHP-01923,2000,2000,52,0,0,52,2.6,Temperature Risk
SHP-03968,2000,2000,52,0,0,52,2.6,Temperature Risk


In [0]:

# Summarize cold-chain risk across all shipments.

risk_overview_df = (
    shipment_risk_df
    .groupBy("risk_status")
    .agg(
        F.countDistinct("shipment_id").alias("shipment_count")
    )
)

display(risk_overview_df.orderBy(F.col("shipment_count").desc()))


risk_status,shipment_count
Temperature Risk,1470


In [0]:

# Check shipment coverage and temperature-limit availability in the Gold dataset.

print("Distinct shipments in risk analysis:", shipment_risk_df.select("shipment_id").distinct().count())

print(
    "Shipments with missing temperature limits:",
    shipment_risk_df
    .filter(F.col("temperature_limits_missing") == 1)
    .select("shipment_id")
    .distinct()
    .count()
)

print(
    "Shipments with no recorded temperature risk:",
    shipment_risk_df
    .filter(F.col("out_of_range_readings") == 0)
    .select("shipment_id")
    .distinct()
    .count()
)


Distinct shipments in risk analysis: 1470
Shipments with missing temperature limits: 0
Shipments with no recorded temperature risk: 0


In [0]:

# Build a complete shipment risk report, including shipments absent from temperature readings.

all_shipments_df = (
    spark.read.format("delta")
    .load(silver_paths["shipments"])
    .select(
        "shipment_id",
        "required_temp_min_c",
        "required_temp_max_c"
    )
)

shipment_risk_complete_df = (
    all_shipments_df
    .join(shipment_risk_df, on="shipment_id", how="left")
    .withColumn(
        "risk_status",
        F.when(
            F.col("total_readings").isNull(),
            "No Temperature Readings"
        ).when(
            F.col("required_temp_min_c").isNull()
            | F.col("required_temp_max_c").isNull(),
            "Temperature Limits Missing"
        ).when(
            F.col("out_of_range_readings") > 0,
            "Temperature Risk"
        ).otherwise("Within Limits")
    )
)

display(
    shipment_risk_complete_df
    .groupBy("risk_status")
    .agg(
        F.countDistinct("shipment_id").alias("shipment_count")
    )
    .orderBy(F.col("shipment_count").desc())
)


risk_status,shipment_count
No Temperature Readings,3530
Temperature Risk,1470


In [0]:

# Investigate shipment monitoring coverage using saved Silver and Gold datasets.

plan_coverage_df = (
    spark.read.format("delta")
    .load(silver_paths["trip_plans"])
    .select("shipment_id", "vehicle_id", "trip_id")
    .dropDuplicates(["shipment_id"])
)

coverage_check_df = (
    all_shipments_df
    .join(plan_coverage_df, on="shipment_id", how="left")
    .join(
        shipment_risk_df.select("shipment_id").distinct()
        .withColumn("has_temperature_readings", F.lit(True)),
        on="shipment_id",
        how="left"
    )
    .withColumn(
        "monitoring_coverage",
        F.when(
            F.col("trip_id").isNull(),
            "No Trip Plan"
        ).when(
            F.col("has_temperature_readings").isNull(),
            "No Temperature Readings"
        ).otherwise("Has Temperature Readings")
    )
)

display(
    coverage_check_df
    .groupBy("monitoring_coverage")
    .agg(F.countDistinct("shipment_id").alias("shipment_count"))
    .orderBy(F.col("shipment_count").desc())
)


monitoring_coverage,shipment_count
No Temperature Readings,3530
Has Temperature Readings,1470


In [0]:

# Check whether shipments without cold-chain readings are linked to vehicles with telemetry.

telemetry_vehicle_df = (
    spark.read.format("delta")
    .load(silver_paths["telemetry_stream"])
    .select("vehicle_id")
    .distinct()
    .withColumn("vehicle_has_telemetry", F.lit(True))
)

shipment_vehicle_coverage_df = (
    plan_coverage_df
    .join(telemetry_vehicle_df, on="vehicle_id", how="left")
    .join(
        shipment_risk_df.select("shipment_id").distinct()
        .withColumn("has_cold_chain_readings", F.lit(True)),
        on="shipment_id",
        how="left"
    )
    .withColumn(
        "coverage_status",
        F.when(
            F.col("vehicle_has_telemetry").isNull(),
            "Vehicle Has No Telemetry"
        ).when(
            F.col("has_cold_chain_readings").isNull(),
            "Vehicle Has Telemetry, But No Cold-Chain Readings"
        ).otherwise("Has Cold-Chain Readings")
    )
)

display(
    shipment_vehicle_coverage_df
    .groupBy("coverage_status")
    .agg(F.countDistinct("shipment_id").alias("shipment_count"))
    .orderBy(F.col("shipment_count").desc())
)


coverage_status,shipment_count
"Vehicle Has Telemetry, But No Cold-Chain Readings",3530
Has Cold-Chain Readings,1470


In [0]:

# Check shipment-level telemetry coverage and missing cargo temperatures.

telemetry_shipment_check_df = (
    spark.read.format("delta")
    .load(silver_paths["telemetry_stream"])
    .select("shipment_id", "cargo_temp_c")
    .groupBy("shipment_id")
    .agg(
        F.count("*").alias("telemetry_events"),
        F.count("cargo_temp_c").alias("events_with_cargo_temp"),
        F.sum(
            F.when(F.col("cargo_temp_c").isNull(), 1).otherwise(0)
        ).alias("events_missing_cargo_temp")
    )
)

plan_shipment_ids_df = (
    spark.read.format("delta")
    .load(silver_paths["trip_plans"])
    .select("shipment_id")
    .distinct()
)

shipment_telemetry_coverage_df = (
    plan_shipment_ids_df
    .join(telemetry_shipment_check_df, on="shipment_id", how="left")
    .withColumn(
        "telemetry_status",
        F.when(
            F.col("telemetry_events").isNull(),
            "No Events Linked to Shipment"
        ).when(
            F.col("events_with_cargo_temp") == 0,
            "Events Exist, Cargo Temperature Missing"
        ).when(
            F.col("events_with_cargo_temp") < F.col("telemetry_events"),
            "Partial Cargo Temperature Coverage"
        ).otherwise("Cargo Temperature Available")
    )
)

display(
    shipment_telemetry_coverage_df
    .groupBy("telemetry_status")
    .agg(F.countDistinct("shipment_id").alias("shipment_count"))
    .orderBy(F.col("shipment_count").desc())
)


telemetry_status,shipment_count
"Events Exist, Cargo Temperature Missing",3530
Cargo Temperature Available,1470


### Cold Chain Data Quality Findings

**Objective:** Evaluate cold-chain monitoring coverage and identify data quality limitations.

**Findings:**
- Total shipments analyzed: 5,000.
- Shipments with available cargo temperature readings: 1,470 (29.4%).
- Shipments with telemetry events but all cargo temperature readings missing: 3,530 (70.6%).
- The missing temperature readings prevent reliable compliance evaluation for these 3,530 shipments.

**Business Impact:**
The current cold-chain compliance KPI represents only readings with available temperature limits and temperature measurements. It should not be interpreted as compliance across all shipments.

**Recommendation:**
Investigate why `cargo_temp_c` is missing for these shipments, validate the upstream telemetry generation and ingestion process, and monitor temperature-data completeness as a separate data quality KPI.

**Limitation:**
Missing temperature readings do not prove that a cold-chain incident occurred. These shipments should be classified as not evaluable until sufficient data is available.

In [0]:

# Validate the four saved Gold Delta datasets using metadata only.

from delta.tables import DeltaTable

gold_paths = {
    "trip_summary": storage_path + "gold/trip_summary/",
    "route_daily_performance": storage_path + "gold/route_daily_performance/",
    "vehicle_daily_scorecard": storage_path + "gold/vehicle_daily_scorecard/",
    "cold_chain_daily_summary": storage_path + "gold/cold_chain_daily_summary/"
}

gold_validation_results = []

for dataset_name, dataset_path in gold_paths.items():
    detail_df = DeltaTable.forPath(spark, dataset_path).detail()

    metadata = detail_df.select(
        "format", "numFiles", "sizeInBytes"
    ).first()

    gold_validation_results.append((
        dataset_name,
        metadata["format"],
        metadata["numFiles"],
        round(metadata["sizeInBytes"] / (1024 * 1024), 3),
        "PASS"
    ))

gold_validation_df = spark.createDataFrame(
    gold_validation_results,
    [
        "dataset_name",
        "format",
        "num_files",
        "size_mb",
        "validation_status"
    ]
)

display(gold_validation_df)


dataset_name,format,num_files,size_mb,validation_status
trip_summary,delta,1,0.113,PASS
route_daily_performance,delta,1,0.003,PASS
vehicle_daily_scorecard,delta,1,0.349,PASS
cold_chain_daily_summary,delta,1,0.057,PASS


In [0]:

# Calculate business KPIs from saved Gold datasets and the existing shipment risk analysis.

trip_summary_kpi_df = (
    spark.read.format("delta").load(gold_paths["trip_summary"])
    .agg(
        F.countDistinct("trip_id").alias("total_trips"),
        F.countDistinct(
            F.when(F.col("operational_status") == "completed", F.col("trip_id"))
        ).alias("recorded_completed_trips"),
        F.countDistinct(
            F.when(F.col("actual_start").isNotNull(), F.col("trip_id"))
        ).alias("trips_with_actual_start"),
        F.countDistinct(
            F.when(F.col("actual_end").isNotNull(), F.col("trip_id"))
        ).alias("trips_with_actual_end")
    )
)

cold_chain_kpi_df = (
    spark.read.format("delta").load(gold_paths["cold_chain_daily_summary"])
    .agg(
        F.sum("readings_with_known_limits").alias("evaluable_readings"),
        F.sum("readings_above_required_max").alias("readings_above_max"),
        F.sum("readings_below_required_min").alias("readings_below_min")
    )
    .withColumn(
        "compliant_readings",
        F.col("evaluable_readings")
        - F.col("readings_above_max")
        - F.col("readings_below_min")
    )
    .withColumn(
        "temperature_compliance_pct",
        F.round(
            F.col("compliant_readings") / F.col("evaluable_readings") * 100,
            2
        )
    )
)

print("TRIP KPIs")
display(trip_summary_kpi_df)

print("COLD-CHAIN KPIs")
display(cold_chain_kpi_df)

print("SHIPMENT MONITORING COVERAGE")
display(
    shipment_risk_complete_df
    .groupBy("risk_status")
    .agg(F.countDistinct("shipment_id").alias("shipment_count"))
    .orderBy(F.col("shipment_count").desc())
)


TRIP KPIs


total_trips,recorded_completed_trips,trips_with_actual_start,trips_with_actual_end
5000,0,5000,0


COLD-CHAIN KPIs


evaluable_readings,readings_above_max,readings_below_min,compliant_readings,temperature_compliance_pct
2940000,52660,0,2887340,98.21


SHIPMENT MONITORING COVERAGE


risk_status,shipment_count
No Temperature Readings,3530
Temperature Risk,1470
